# Day 20: Structured Outputs with Pydantic

**Name:** Areeba Amjad  

**Date:** 25 August 2026 

**Topic:** Structured Outputs with Pydantic  


## Objective

Today I will learn how to use Pydantic for structured and validated data.

### Topics Covered

- What is Pydantic?
- BaseModel
- Typed fields
- Automatic validation
- Automatic parsing
- JSON serialization
- Optional and Union types
- List fields
- Custom validators
- Validation errors
- LangChain structured outputs
- `with_structured_output()`
- `JsonOutputParser`
- Why structured outputs are useful

In [1]:
%pip install -U pydantic

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pydantic import BaseModel, Field, field_validator, ValidationError
from typing import List, Optional, Union

## What is Pydantic?

Pydantic is a Python data validation library that uses Python type hints
to validate and parse data.

It helps developers make sure that incoming data follows the expected
structure and data types.

### Main Features

1. Data validation
2. Type checking
3. Automatic parsing/conversion
4. Custom validation
5. JSON serialization
6. Clear validation errors

Pydantic is commonly used in APIs, LLM applications, FastAPI,
data pipelines, and structured AI outputs.

In [3]:
class Student(BaseModel):
    name: str
    age: int
    department: str

In [4]:
student = Student(
    name="Areeba",
    age=21,
    department="Data Science"
)

student

Student(name='Areeba', age=21, department='Data Science')

In [5]:
print("Name:", student.name)
print("Age:", student.age)
print("Department:", student.department)

Name: Areeba
Age: 21
Department: Data Science


In [6]:
try:
    student = Student(
        name="Areeba",
        age="twenty one",
        department="Data Science"
    )
except ValidationError as e:
    print(e)

1 validation error for Student
age
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='twenty one', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/int_parsing


In [7]:
class Product(BaseModel):
    name: str
    price: float
    quantity: int

In [8]:
product = Product(
    name="Laptop",
    price="150000",
    quantity="2"
)

print(product)
print(type(product.price))
print(type(product.quantity))

name='Laptop' price=150000.0 quantity=2
<class 'float'>
<class 'int'>


In [9]:
class User(BaseModel):
    name: str
    age: int
    skills: List[str]
    email: Optional[str] = None
    user_id: Union[int, str]

In [10]:
user = User(
    name="Areeba",
    age=21,
    skills=["Python", "Machine Learning", "LangChain"],
    user_id=101
)

print(user)

name='Areeba' age=21 skills=['Python', 'Machine Learning', 'LangChain'] email=None user_id=101


In [11]:
user2 = User(
    name="Ali",
    age=22,
    skills=["Python"],
    user_id="USER-102"
)

print(user2)
print("Email:", user2.email)

name='Ali' age=22 skills=['Python'] email=None user_id='USER-102'
Email: None


## Common Pydantic Field Types

| Type | Purpose |
|------|---------|
| `str` | Text |
| `int` | Integer |
| `float` | Decimal number |
| `bool` | True/False |
| `List[str]` | List of strings |
| `Optional[str]` | Value can be string or None |
| `Union[int, str]` | Value can be integer or string |

These type hints make the expected data structure explicit.

In [12]:
class Employee(BaseModel):
    name: str = Field(min_length=2)
    age: int = Field(gt=18, lt=60)
    salary: float = Field(gt=0)

In [13]:
employee = Employee(
    name="Areeba",
    age=21,
    salary=80000
)

print(employee)

name='Areeba' age=21 salary=80000.0


In [14]:
try:
    employee = Employee(
        name="A",
        age=15,
        salary=-5000
    )
except ValidationError as e:
    print(e)

3 validation errors for Employee
name
  String should have at least 2 characters [type=string_too_short, input_value='A', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/string_too_short
age
  Input should be greater than 18 [type=greater_than, input_value=15, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/greater_than
salary
  Input should be greater than 0 [type=greater_than, input_value=-5000, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/greater_than


In [15]:
class StudentProfile(BaseModel):
    name: str
    age: int
    email: str

    @field_validator("age")
    @classmethod
    def validate_age(cls, value):
        if value < 18:
            raise ValueError("Student must be at least 18 years old")
        return value

    @field_validator("email")
    @classmethod
    def validate_email(cls, value):
        if "@" not in value:
            raise ValueError("Invalid email address")
        return value

In [16]:
try:
    student = StudentProfile(
        name="Areeba",
        age=21,
        email="areeba@example.com"
    )

    print(student)

except ValidationError as e:
    print(e)

name='Areeba' age=21 email='areeba@example.com'


In [17]:
try:
    student = StudentProfile(
        name="Areeba",
        age=16,
        email="invalid-email"
    )

except ValidationError as e:
    print(e)

2 validation errors for StudentProfile
age
  Value error, Student must be at least 18 years old [type=value_error, input_value=16, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error
email
  Value error, Invalid email address [type=value_error, input_value='invalid-email', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error


In [18]:
student = Student(
    name="Areeba",
    age=21,
    department="Data Science"
)

print(student.model_dump())

{'name': 'Areeba', 'age': 21, 'department': 'Data Science'}


In [19]:
json_data = student.model_dump_json()

print(json_data)

{"name":"Areeba","age":21,"department":"Data Science"}


In [20]:
student_data = {
    "name": "Areeba",
    "age": 21,
    "department": "Data Science"
}

student = Student.model_validate(student_data)

print(student)

name='Areeba' age=21 department='Data Science'


In [21]:
class Movie(BaseModel):
    title: str
    year: int
    genre: List[str]
    rating: float
    description: Optional[str] = None

In [22]:
class Movie(BaseModel):
    title: str
    year: int
    genre: List[str]
    rating: float
    description: Optional[str] = None

In [23]:
movie = Movie(
    title="Inception",
    year=2010,
    genre=["Sci-Fi", "Thriller"],
    rating=8.8,
    description="A science-fiction thriller about dreams."
)

print(movie)

title='Inception' year=2010 genre=['Sci-Fi', 'Thriller'] rating=8.8 description='A science-fiction thriller about dreams.'


In [24]:
print(movie.model_dump_json(indent=2))

{
  "title": "Inception",
  "year": 2010,
  "genre": [
    "Sci-Fi",
    "Thriller"
  ],
  "rating": 8.8,
  "description": "A science-fiction thriller about dreams."
}


## Why Structured Outputs?

LLMs normally generate free-form text.

For example:

"The movie is Inception. It was released in 2010.
It is a science-fiction thriller and has a rating of 8.8."

A program has to manually extract this information.

With structured output, the expected schema can be:

{
    "title": "Inception",
    "year": 2010,
    "genre": ["Sci-Fi", "Thriller"],
    "rating": 8.8
}

### Benefits

- Reliable data format
- Type safety
- Easier programmatic processing
- Better error handling
- Easier API integration
- Less manual parsing
- Consistent outputs

In [1]:
%pip install -U langchain langchain-core

Note: you may need to restart the kernel to use updated packages.


## LangChain Structured Output

LangChain provides mechanisms for converting LLM responses into
structured data.

One important method is:

`with_structured_output()`

It can be used with supported chat models to specify a schema such as
a Pydantic model.

The model response can then be returned in a structured format rather
than plain text.

Another approach is using output parsers such as:

- `JsonOutputParser`
- Other LangChain output parsers

These help convert model output into data that applications can process.

In [2]:
from pydantic import BaseModel, Field
from typing import List


class MovieRecommendation(BaseModel):
    title: str = Field(description="Movie title")
    year: int = Field(description="Release year")
    genres: List[str] = Field(description="Movie genres")
    rating: float = Field(description="Movie rating")

In [3]:
# Example:
# structured_llm = llm.with_structured_output(MovieRecommendation)

# response = structured_llm.invoke(
#     "Recommend a science fiction movie."
# )

# print(response)


In [4]:
from langchain_core.output_parsers import JsonOutputParser

In [5]:
parser = JsonOutputParser(pydantic_object=MovieRecommendation)

print(parser.get_format_instructions())

STRICT OUTPUT FORMAT:
- Return only the JSON value that conforms to the schema. Do not include any additional text, explanations, headings, or separators.
- Do not wrap the JSON in Markdown or code fences (no ``` or ```json).
- Do not prepend or append any text (e.g., do not write "Here is the JSON:").
- The response must be a single top-level JSON value exactly as required by the schema (object/array/etc.), with no trailing commas or comments.

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]} the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema (shown in a code block for readability only — do not include any backticks or Markdown in your output):


In [6]:
json_response = """
{
    "title": "Interstellar",
    "year": 2014,
    "genres": ["Science Fiction", "Drama"],
    "rating": 8.7
}
"""

In [7]:
parsed_data = parser.parse(json_response)

print(parsed_data)
print(type(parsed_data))

{'title': 'Interstellar', 'year': 2014, 'genres': ['Science Fiction', 'Drama'], 'rating': 8.7}
<class 'dict'>


In [8]:
validated_movie = MovieRecommendation.model_validate(parsed_data)

print(validated_movie)

title='Interstellar' year=2014 genres=['Science Fiction', 'Drama'] rating=8.7


In [9]:
print(validated_movie.model_dump())

{'title': 'Interstellar', 'year': 2014, 'genres': ['Science Fiction', 'Drama'], 'rating': 8.7}


In [11]:
from pydantic import ValidationError

In [12]:
invalid_movie = {
    "title": "Example Movie",
    "year": "not-a-year",
    "genres": ["Drama"],
    "rating": 8.5
}

try:
    movie = MovieRecommendation.model_validate(invalid_movie)
    print(movie)

except ValidationError as e:
    print("Validation Error:")
    print(e)

Validation Error:
1 validation error for MovieRecommendation
year
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='not-a-year', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/int_parsing


In [14]:
from typing import List, Optional
from pydantic import BaseModel, field_validator, ValidationError

In [15]:
class JobPosting(BaseModel):
    title: str
    company: str
    skills: List[str]
    experience_years: int
    remote: bool
    salary: Optional[float] = None

    @field_validator("experience_years")
    @classmethod
    def validate_experience(cls, value):
        if value < 0:
            raise ValueError("Experience cannot be negative")
        return value

In [16]:
job = JobPosting(
    title="Junior Data Scientist",
    company="AI Solutions",
    skills=["Python", "Machine Learning", "SQL"],
    experience_years=1,
    remote=True,
    salary=100000
)

print(job)

title='Junior Data Scientist' company='AI Solutions' skills=['Python', 'Machine Learning', 'SQL'] experience_years=1 remote=True salary=100000.0


In [18]:
print(job.model_dump_json(indent=2))

{
  "title": "Junior Data Scientist",
  "company": "AI Solutions",
  "skills": [
    "Python",
    "Machine Learning",
    "SQL"
  ],
  "experience_years": 1,
  "remote": true,
  "salary": 100000.0
}


# Research Summary

## 1. Pydantic

Pydantic is a Python library for validating and parsing data using
Python type annotations.

## 2. BaseModel

`BaseModel` is the main Pydantic class used to define structured data
models.

It provides:

- Validation
- Type conversion
- Serialization
- Schema generation
- Clear validation errors

## 3. Field Types

Pydantic supports Python typing features such as:

- `str`
- `int`
- `float`
- `bool`
- `List`
- `Optional`
- `Union`

It also supports constraints through `Field()` and custom validation
through validators.

## 4. LangChain Integration

LangChain can work with structured schemas through methods such as
`with_structured_output()`.

Output parsers such as `JsonOutputParser` can also help transform
model-generated output into structured Python data.

## 5. Why Structured Outputs?

Structured outputs are important because LLMs normally produce
unstructured natural language.

Using a predefined schema makes the output:

- More predictable
- Easier to validate
- Easier to parse
- Safer to integrate into applications
- Easier to store in databases
- Easier to pass between software components

# Research Sources

### AI Research

1. ChatGPT
   - Used to understand Pydantic fundamentals, BaseModel,
     validation, structured outputs, and LangChain integration.

2. Google Gemini
   - Used to compare explanations of Pydantic validation and
     structured LLM outputs.

3. Claude
   - Used to cross-check concepts related to type validation,
     JSON parsing, and structured outputs.

### Articles / Documentation

4. Pydantic Documentation
   - https://docs.pydantic.dev/

5. LangChain Documentation
   - https://python.langchain.com/

In [19]:
class StudentResult(BaseModel):
    name: str
    age: int
    subjects: List[str]
    cgpa: float = Field(ge=0, le=4)
    email: Optional[str] = None

    @field_validator("age")
    @classmethod
    def check_age(cls, value):
        if value < 18:
            raise ValueError("Age must be 18 or above")
        return value

In [20]:
student_result = StudentResult(
    name="Areeba",
    age=21,
    subjects=[
        "Python",
        "Machine Learning",
        "Data Science",
        "LLM"
    ],
    cgpa=3.2,
    email="areeba@example.com"
)

print(student_result)

name='Areeba' age=21 subjects=['Python', 'Machine Learning', 'Data Science', 'LLM'] cgpa=3.2 email='areeba@example.com'


In [21]:
result_dict = student_result.model_dump()

print(result_dict)

{'name': 'Areeba', 'age': 21, 'subjects': ['Python', 'Machine Learning', 'Data Science', 'LLM'], 'cgpa': 3.2, 'email': 'areeba@example.com'}


In [22]:
result_json = student_result.model_dump_json(indent=2)

print(result_json)

{
  "name": "Areeba",
  "age": 21,
  "subjects": [
    "Python",
    "Machine Learning",
    "Data Science",
    "LLM"
  ],
  "cgpa": 3.2,
  "email": "areeba@example.com"
}


In [23]:
try:
    invalid_result = StudentResult(
        name="Test Student",
        age=16,
        subjects=["Python"],
        cgpa=5.0
    )

except ValidationError as e:
    print("Validation failed successfully!")
    print(e)

Validation failed successfully!
2 validation errors for StudentResult
age
  Value error, Age must be 18 or above [type=value_error, input_value=16, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error
cgpa
  Input should be less than or equal to 4 [type=less_than_equal, input_value=5.0, input_type=float]
    For further information visit https://errors.pydantic.dev/2.13/v/less_than_equal


# Day 20 Learning Summary

Today I learned how Pydantic can be used to create reliable structured
data models in Python.

### Key Learnings

- Pydantic performs data validation using type hints.
- `BaseModel` is used to define structured models.
- Pydantic supports types such as `str`, `int`, `List`, `Optional`,
  and `Union`.
- `Field()` can be used to add constraints.
- Custom validators can implement application-specific rules.
- Pydantic can automatically parse compatible input values.
- `model_dump()` converts models into dictionaries.
- `model_dump_json()` converts models into JSON.
- `ValidationError` provides useful error information.
- LangChain supports structured LLM outputs.
- `with_structured_output()` can connect LLM responses to schemas.
- `JsonOutputParser` can parse JSON responses.
- Structured outputs make LLM applications more reliable and easier
  to integrate with software systems.

## Conclusion

Pydantic provides a reliable way to define and validate structured
data, while LangChain can use these schemas to make LLM outputs more
predictable and useful for real-world applications.


### Task 1: Pydantic Model Suite
- Create Person model
- Create Product model
- Add validation
- Test valid and invalid data

### Task 2: LLM → Structured Data Pipeline
- Define Article model
- Use `with_structured_output()`
- Process raw article text
- Validate structured output
- Handle validation errors
- Process batch articles
- Export JSON dataset

### Task 3: Multi-Entity Extraction System
- Create nested Company and Employee models
- Extract companies and employees
- Build company → employee → skills structure
- Validate extracted data
- Export JSON
- Calculate extraction accuracy

In [25]:
%pip install -U pydantic

Note: you may need to restart the kernel to use updated packages.


In [26]:
%pip install -U langchain langchain-core

Note: you may need to restart the kernel to use updated packages.


In [27]:
from typing import List, Optional, Union
from pydantic import BaseModel, Field, field_validator, ValidationError
import json
import re

In [28]:
class Person(BaseModel):
    name: str
    age: int
    email: str
    hobbies: List[str]

    @field_validator("age")
    @classmethod
    def validate_age(cls, value):
        if value <= 0:
            raise ValueError("Age must be greater than 0")
        return value

    @field_validator("email")
    @classmethod
    def validate_email(cls, value):
        email_pattern = r"^[^@\s]+@[^@\s]+\.[^@\s]+$"

        if not re.match(email_pattern, value):
            raise ValueError("Invalid email format")

        return value

In [29]:
class Product(BaseModel):
    id: int
    name: str
    price: float
    in_stock: bool
    tags: List[str]

    @field_validator("price")
    @classmethod
    def validate_price(cls, value):
        if value < 0:
            raise ValueError("Price must be greater than or equal to 0")
        return value

In [30]:
person = Person(
    name="Areeba",
    age=21,
    email="areeba@example.com",
    hobbies=["Reading", "Coding", "Photography"]
)

print(person)

name='Areeba' age=21 email='areeba@example.com' hobbies=['Reading', 'Coding', 'Photography']


In [31]:
product = Product(
    id=101,
    name="Laptop",
    price=150000,
    in_stock=True,
    tags=["electronics", "computer", "laptop"]
)

print(product)

id=101 name='Laptop' price=150000.0 in_stock=True tags=['electronics', 'computer', 'laptop']


In [32]:
invalid_person = {
    "name": "Test User",
    "age": -5,
    "email": "invalid-email",
    "hobbies": ["Gaming"]
}

try:
    person = Person.model_validate(invalid_person)
    print(person)

except ValidationError as e:
    print("Validation Error:")
    print(e)

Validation Error:
2 validation errors for Person
age
  Value error, Age must be greater than 0 [type=value_error, input_value=-5, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error
email
  Value error, Invalid email format [type=value_error, input_value='invalid-email', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error


In [33]:
invalid_product = {
    "id": 102,
    "name": "Phone",
    "price": -5000,
    "in_stock": True,
    "tags": ["electronics"]
}

try:
    product = Product.model_validate(invalid_product)
    print(product)

except ValidationError as e:
    print("Validation Error:")
    print(e)

Validation Error:
1 validation error for Product
price
  Value error, Price must be greater than or equal to 0 [type=value_error, input_value=-5000, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error


In [34]:
print(person.model_dump())

{'name': 'Areeba', 'age': 21, 'email': 'areeba@example.com', 'hobbies': ['Reading', 'Coding', 'Photography']}


In [35]:
print(product.model_dump_json(indent=2))

{
  "id": 101,
  "name": "Laptop",
  "price": 150000.0,
  "in_stock": true,
  "tags": [
    "electronics",
    "computer",
    "laptop"
  ]
}


## Task 1 Result

Task 1 successfully demonstrates:

- Pydantic `BaseModel`
- Typed fields
- `List[str]`
- Custom validators
- Email validation
- Age validation
- Price validation
- Validation errors
- Dictionary serialization
- JSON serialization

Valid data is accepted, while invalid data raises `ValidationError`.

## TASK 2 — LLM → Structured Data Pipeline

In [36]:
class Article(BaseModel):
    title: str
    author: str
    published_date: str
    summary: str
    tags: List[str]

In [37]:
article = Article(
    title="Artificial Intelligence in Modern Applications",
    author="John Smith",
    published_date="2026-08-25",
    summary="Artificial intelligence is increasingly being used in modern software applications.",
    tags=["AI", "Machine Learning", "Technology"]
)

print(article)

title='Artificial Intelligence in Modern Applications' author='John Smith' published_date='2026-08-25' summary='Artificial intelligence is increasingly being used in modern software applications.' tags=['AI', 'Machine Learning', 'Technology']


In [38]:
article_json = article.model_dump_json(indent=2)

print(article_json)

{
  "title": "Artificial Intelligence in Modern Applications",
  "author": "John Smith",
  "published_date": "2026-08-25",
  "summary": "Artificial intelligence is increasingly being used in modern software applications.",
  "tags": [
    "AI",
    "Machine Learning",
    "Technology"
  ]
}


## LangChain Structured Output

LangChain's `with_structured_output()` allows an LLM to return data
according to a predefined schema.

Instead of receiving free-form text, the application can request an
`Article` object.

Conceptually:

model
   ↓
with_structured_output(Article)
   ↓
Article object
   ↓
Pydantic validation
   ↓
Structured JSON

In [39]:
from langchain_core.prompts import ChatPromptTemplate

In [40]:
# Example only:
#
# from langchain_openai import ChatOpenAI
#
# model = ChatOpenAI(
#     model="YOUR_MODEL_NAME",
#     temperature=0
# )

In [41]:
# Once your LLM is configured:

# chain = model.with_structured_output(Article)

# response = chain.invoke(
#     """
#     Artificial intelligence is transforming software development.
#     The article was written by Sarah Khan and published on
#     August 25, 2026. It discusses AI applications in modern
#     software development.
#     """
# )

# print(response)

In [43]:
raw_article = """
Artificial Intelligence in Modern Software Development

Written by Sarah Khan
Published: 2026-08-25

Artificial intelligence is transforming modern software development.
AI tools can help developers write code, analyze errors, automate
testing, and improve productivity.

The article discusses artificial intelligence, software engineering,
automation, and developer productivity.
"""

In [44]:
structured_article_data = {
    "title": "Artificial Intelligence in Modern Software Development",
    "author": "Sarah Khan",
    "published_date": "2026-08-25",
    "summary": "AI is transforming modern software development by helping developers write code, analyze errors, automate testing, and improve productivity.",
    "tags": [
        "Artificial Intelligence",
        "Software Development",
        "Automation"
    ]
}

In [45]:
structured_article_data = {
    "title": "Artificial Intelligence in Modern Software Development",
    "author": "Sarah Khan",
    "published_date": "2026-08-25",
    "summary": "AI is transforming modern software development by helping developers write code, analyze errors, automate testing, and improve productivity.",
    "tags": [
        "Artificial Intelligence",
        "Software Development",
        "Automation"
    ]
}

In [46]:
invalid_article = {
    "title": "AI Article",
    "author": "Sarah Khan",
    "published_date": "2026-08-25",
    "summary": "An article about artificial intelligence."
    # tags intentionally missing
}

try:
    article = Article.model_validate(invalid_article)
    print(article)

except ValidationError as e:
    print("Validation Error:")
    print(e)

Validation Error:
1 validation error for Article
tags
  Field required [type=missing, input_value={'title': 'AI Article', '...tificial intelligence.'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing


In [47]:
def validate_article(data, max_retries=2):
    for attempt in range(max_retries + 1):

        try:
            article = Article.model_validate(data)

            print(f"Validation successful on attempt {attempt + 1}")
            return article

        except ValidationError as e:

            print(f"Attempt {attempt + 1} failed")
            print(e)

            # Default value for missing tags
            if "tags" not in data:
                data["tags"] = ["Uncategorized"]

    print("Could not validate article.")
    return None
    

In [48]:
article_data = {
    "title": "AI in Healthcare",
    "author": "Ali Ahmed",
    "published_date": "2026-08-25",
    "summary": "Artificial intelligence applications in healthcare."
}

result = validate_article(article_data)

print(result)

Attempt 1 failed
1 validation error for Article
tags
  Field required [type=missing, input_value={'title': 'AI in Healthca...cations in healthcare.'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing
Validation successful on attempt 2
title='AI in Healthcare' author='Ali Ahmed' published_date='2026-08-25' summary='Artificial intelligence applications in healthcare.' tags=['Uncategorized']


In [49]:
articles_data = [
    {
        "title": f"Artificial Intelligence Article {i}",
        "author": f"Author {i}",
        "published_date": "2026-08-25",
        "summary": f"This is a summary of artificial intelligence article {i}.",
        "tags": ["AI", "Technology"]
    }
    for i in range(1, 11)
]

len(articles_data)

10

In [50]:
validated_articles = []

for data in articles_data:
    try:
        article = Article.model_validate(data)
        validated_articles.append(article)

    except ValidationError as e:
        print("Validation failed:")
        print(e)

print("Total valid articles:", len(validated_articles))

Total valid articles: 10


In [51]:
json_dataset = [
    article.model_dump()
    for article in validated_articles
]

print(json.dumps(json_dataset, indent=2))

[
  {
    "title": "Artificial Intelligence Article 1",
    "author": "Author 1",
    "published_date": "2026-08-25",
    "summary": "This is a summary of artificial intelligence article 1.",
    "tags": [
      "AI",
      "Technology"
    ]
  },
  {
    "title": "Artificial Intelligence Article 2",
    "author": "Author 2",
    "published_date": "2026-08-25",
    "summary": "This is a summary of artificial intelligence article 2.",
    "tags": [
      "AI",
      "Technology"
    ]
  },
  {
    "title": "Artificial Intelligence Article 3",
    "author": "Author 3",
    "published_date": "2026-08-25",
    "summary": "This is a summary of artificial intelligence article 3.",
    "tags": [
      "AI",
      "Technology"
    ]
  },
  {
    "title": "Artificial Intelligence Article 4",
    "author": "Author 4",
    "published_date": "2026-08-25",
    "summary": "This is a summary of artificial intelligence article 4.",
    "tags": [
      "AI",
      "Technology"
    ]
  },
  {
    "title

In [52]:
with open("articles_dataset.json", "w", encoding="utf-8") as f:
    json.dump(json_dataset, f, indent=2, ensure_ascii=False)

print("articles_dataset.json created successfully!")

articles_dataset.json created successfully!


## Task 2 Result

The LLM → Structured Data pipeline consists of:

Raw Article Text
       ↓
LLM
       ↓
with_structured_output(Article)
       ↓
Pydantic Article Model
       ↓
Validation
       ↓
Structured JSON
       ↓
Dataset

The notebook also demonstrates:

- Article schema
- Pydantic validation
- Validation error handling
- Retry/default-value logic
- Batch processing
- 10 article records
- JSON dataset generation

The live LLM call requires a configured LangChain chat model and API.
The remaining structured-data pipeline can be tested locally.

## TASK 3 — Multi-Entity Extraction System

In [54]:
class Employee(BaseModel):
    name: str
    title: str
    department: str
    skills: List[str]

In [55]:
class Company(BaseModel):
    name: str
    location: str
    employees: List[Employee]

In [56]:
company = Company(
    name="TechNova",
    location="Lahore, Pakistan",
    employees=[
        Employee(
            name="Ali Khan",
            title="Senior Data Scientist",
            department="AI",
            skills=["Python", "Machine Learning", "SQL"]
        ),
        Employee(
            name="Sara Ahmed",
            title="Software Engineer",
            department="Engineering",
            skills=["Python", "FastAPI", "Docker"]
        )
    ]
)

print(company)

name='TechNova' location='Lahore, Pakistan' employees=[Employee(name='Ali Khan', title='Senior Data Scientist', department='AI', skills=['Python', 'Machine Learning', 'SQL']), Employee(name='Sara Ahmed', title='Software Engineer', department='Engineering', skills=['Python', 'FastAPI', 'Docker'])]


In [57]:
print(company.model_dump_json(indent=2))

{
  "name": "TechNova",
  "location": "Lahore, Pakistan",
  "employees": [
    {
      "name": "Ali Khan",
      "title": "Senior Data Scientist",
      "department": "AI",
      "skills": [
        "Python",
        "Machine Learning",
        "SQL"
      ]
    },
    {
      "name": "Sara Ahmed",
      "title": "Software Engineer",
      "department": "Engineering",
      "skills": [
        "Python",
        "FastAPI",
        "Docker"
      ]
    }
  ]
}


In [58]:
company_text = """
TechNova is a technology company based in Lahore, Pakistan.

Ali Khan is a Senior Data Scientist working in the AI department.
His skills include Python, Machine Learning, and SQL.

Sara Ahmed is a Software Engineer in the Engineering department.
She works with Python, FastAPI, and Docker.
"""

In [59]:
structured_company_data = {
    "name": "TechNova",
    "location": "Lahore, Pakistan",
    "employees": [
        {
            "name": "Ali Khan",
            "title": "Senior Data Scientist",
            "department": "AI",
            "skills": [
                "Python",
                "Machine Learning",
                "SQL"
            ]
        },
        {
            "name": "Sara Ahmed",
            "title": "Software Engineer",
            "department": "Engineering",
            "skills": [
                "Python",
                "FastAPI",
                "Docker"
            ]
        }
    ]
}

In [60]:
try:
    validated_company = Company.model_validate(structured_company_data)

    print("Company data validated successfully!")
    print(validated_company)

except ValidationError as e:
    print("Validation Error:")
    print(e)

Company data validated successfully!
name='TechNova' location='Lahore, Pakistan' employees=[Employee(name='Ali Khan', title='Senior Data Scientist', department='AI', skills=['Python', 'Machine Learning', 'SQL']), Employee(name='Sara Ahmed', title='Software Engineer', department='Engineering', skills=['Python', 'FastAPI', 'Docker'])]


## Knowledge Graph Structure

The extracted information can be represented conceptually as:

Company
   │
   ├── Employee
   │      ├── Skill
   │      ├── Skill
   │      └── Skill
   │
   └── Employee
          ├── Skill
          ├── Skill
          └── Skill

Example:

TechNova
 ├── Ali Khan
 │     ├── Python
 │     ├── Machine Learning
 │     └── SQL
 │
 └── Sara Ahmed
       ├── Python
       ├── FastAPI
       └── Docker

In [61]:
def build_knowledge_graph(company):
    graph = {
        "company": company.name,
        "location": company.location,
        "employees": []
    }

    for employee in company.employees:
        employee_data = {
            "name": employee.name,
            "title": employee.title,
            "department": employee.department,
            "skills": employee.skills
        }

        graph["employees"].append(employee_data)

    return graph

In [62]:
knowledge_graph = build_knowledge_graph(validated_company)

print(json.dumps(knowledge_graph, indent=2))

{
  "company": "TechNova",
  "location": "Lahore, Pakistan",
  "employees": [
    {
      "name": "Ali Khan",
      "title": "Senior Data Scientist",
      "department": "AI",
      "skills": [
        "Python",
        "Machine Learning",
        "SQL"
      ]
    },
    {
      "name": "Sara Ahmed",
      "title": "Software Engineer",
      "department": "Engineering",
      "skills": [
        "Python",
        "FastAPI",
        "Docker"
      ]
    }
  ]
}


In [63]:
def find_employees_by_skill(company, skill):
    results = []

    for employee in company.employees:
        if skill.lower() in [s.lower() for s in employee.skills]:
            results.append(employee.name)

    return results

In [64]:
python_developers = find_employees_by_skill(
    validated_company,
    "Python"
)

print("Employees with Python skill:")
print(python_developers)

Employees with Python skill:
['Ali Khan', 'Sara Ahmed']


In [65]:
company_json = validated_company.model_dump()

with open("company_knowledge_graph.json", "w", encoding="utf-8") as f:
    json.dump(company_json, f, indent=2, ensure_ascii=False)

print("company_knowledge_graph.json created successfully!")

company_knowledge_graph.json created successfully!


In [66]:
def validate_companies(companies_data):
    validated = []
    errors = []

    for i, data in enumerate(companies_data):

        try:
            company = Company.model_validate(data)
            validated.append(company)

        except ValidationError as e:
            errors.append({
                "index": i,
                "error": str(e)
            })

    return validated, errors

In [67]:
companies_data = [
    structured_company_data,
    
    {
        "name": "DataWorks",
        "location": "Islamabad, Pakistan",
        "employees": [
            {
                "name": "Hamza Ali",
                "title": "ML Engineer",
                "department": "Machine Learning",
                "skills": ["Python", "TensorFlow", "PyTorch"]
            }
        ]
    }
]

validated_companies, errors = validate_companies(companies_data)

print("Valid companies:", len(validated_companies))
print("Errors:", len(errors))

Valid companies: 2
Errors: 0


In [68]:
ground_truth = {
    "name": "TechNova",
    "location": "Lahore, Pakistan",
    "employees": [
        {
            "name": "Ali Khan",
            "title": "Senior Data Scientist",
            "department": "AI",
            "skills": ["Python", "Machine Learning", "SQL"]
        },
        {
            "name": "Sara Ahmed",
            "title": "Software Engineer",
            "department": "Engineering",
            "skills": ["Python", "FastAPI", "Docker"]
        }
    ]
}

In [69]:
def calculate_accuracy(expected, actual):
    total = 0
    correct = 0

    # Company fields
    for field in ["name", "location"]:
        total += 1

        if expected[field] == actual[field]:
            correct += 1

    # Employee fields
    for expected_emp in expected["employees"]:

        actual_emp = next(
            (
                emp for emp in actual["employees"]
                if emp["name"] == expected_emp["name"]
            ),
            None
        )

        if actual_emp is None:
            total += 4
            continue

        for field in ["name", "title", "department", "skills"]:
            total += 1

            if actual_emp[field] == expected_emp[field]:
                correct += 1

    accuracy = (correct / total) * 100

    return accuracy

In [70]:
actual_data = validated_company.model_dump()

accuracy = calculate_accuracy(
    ground_truth,
    actual_data
)

print(f"Extraction Accuracy: {accuracy:.2f}%")

Extraction Accuracy: 100.00%


In [71]:
wrong_data = validated_company.model_dump()

wrong_data["employees"][0]["department"] = "Engineering"

wrong_accuracy = calculate_accuracy(
    ground_truth,
    wrong_data
)

print(f"Extraction Accuracy: {wrong_accuracy:.2f}%")

Extraction Accuracy: 90.00%


In [72]:
final_output = {
    "companies": [
        company.model_dump()
        for company in validated_companies
    ]
}

with open(
    "multi_entity_extraction.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        final_output,
        f,
        indent=2,
        ensure_ascii=False
    )

print("multi_entity_extraction.json created successfully!")

multi_entity_extraction.json created successfully!


In [73]:
print("========== DAY 20 TASK VERIFICATION ==========")

print("\nTask 1:")
print("✓ Person model")
print("✓ Product model")
print("✓ Email validation")
print("✓ Age validation")
print("✓ Price validation")
print("✓ Invalid data testing")

print("\nTask 2:")
print("✓ Article model")
print("✓ Structured output schema")
print("✓ Validation")
print("✓ Error handling")
print("✓ Retry/default value")
print("✓ Batch processing")
print("✓ 10 article dataset")
print("✓ JSON export")

print("\nTask 3:")
print("✓ Company model")
print("✓ Employee nested model")
print("✓ Skills extraction")
print("✓ Knowledge graph structure")
print("✓ JSON export")
print("✓ Data validation")
print("✓ Extraction accuracy")

print("\nAll Day 20 tasks completed!")

========== DAY 20 TASK VERIFICATION ==========

Task 1:
✓ Person model
✓ Product model
✓ Email validation
✓ Age validation
✓ Price validation
✓ Invalid data testing

Task 2:
✓ Article model
✓ Structured output schema
✓ Validation
✓ Error handling
✓ Retry/default value
✓ Batch processing
✓ 10 article dataset
✓ JSON export

Task 3:
✓ Company model
✓ Employee nested model
✓ Skills extraction
✓ Knowledge graph structure
✓ JSON export
✓ Data validation
✓ Extraction accuracy

All Day 20 tasks completed!
